In [1]:
# Checking if required files are present
import os
for f in ['issues_with_prs.csv', 'no_pr_issues.csv', 'treesitter_index.json']:
    path = f'/content/{f}'
    size = os.path.getsize(path) / (1024*1024)
    print(f'{f}: {size:.1f} MB')

issues_with_prs.csv: 169.0 MB
no_pr_issues.csv: 49.5 MB
treesitter_index.json: 0.5 MB


In [2]:
import torch
print(torch.cuda.get_device_name(0))
print(f'{torch.cuda.get_device_properties(0).total_memory / 1e9:.0f}GB VRAM')

NVIDIA A100-SXM4-80GB
85GB VRAM


In [3]:
# ============================================================
# CELL 1 — INSTALL DEPENDENCIES
# Restart session after this cell completes
# ============================================================
!pip install -q transformers peft accelerate bitsandbytes datasets trl scikit-learn
!pip install -q snowflake-connector-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 48.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.0/105.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.7/14.7 MB 126.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 124.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 12.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pydrive2 1.21.3 requires crypt

In [1]:
# Checking we're fine with bits and bytes
import bitsandbytes as bnb
print(f'bitsandbytes: {bnb.__version__}')

bitsandbytes: 0.49.2


In [2]:
# ============================================================
# CELL 2 — MOUNT GOOGLE DRIVE
# Adapter, JSONL, CSV backup and checkpoints all saved here
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/298B_WB2'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive mounted. Outputs: {DRIVE_ROOT}')

Mounted at /content/drive
Drive mounted. Outputs: /content/drive/MyDrive/298B_WB2


In [7]:
# ============================================================
# COPY CSVs FROM DRIVE TO COLAB WORKING DIRECTORY
# Run this instead of uploading via the upload button
# Copies are instant since Drive is already mounted
# ============================================================
import shutil

shutil.copy(f'{DRIVE_ROOT}/issues_with_prs.csv',  '/content/issues_with_prs.csv')
shutil.copy(f'{DRIVE_ROOT}/no_pr_issues.csv',     '/content/no_pr_issues.csv')
shutil.copy(f'{DRIVE_ROOT}/treesitter_index.json','/content/treesitter_index.json')

print('Files copied to /content/:')
for f in ['issues_with_prs.csv', 'no_pr_issues.csv', 'treesitter_index.json']:
    size = os.path.getsize(f'/content/{f}') / (1024*1024)
    print(f'  {f}: {size:.1f} MB')

Files copied to /content/:
  issues_with_prs.csv: 1399.9 MB
  no_pr_issues.csv: 49.5 MB
  treesitter_index.json: 0.5 MB


In [8]:
# ============================================================
# CELL 3 — CONFIGURATION
# ============================================================

# Tree-sitter index
TREESITTER_INDEX_PATH = '/content/treesitter_index.json'

ISSUES_WITH_PRS_CSV = '/content/issues_with_prs.csv'
NO_PR_ISSUES_CSV    = '/content/no_pr_issues.csv'

# Training targets
YES_TARGET = 2200
NO_TARGET  = 300

# Model
BASE_MODEL = 'Qwen/Qwen2.5-Coder-7B-Instruct'

# Output paths on Drive
OUTPUT_DIR      = f'{DRIVE_ROOT}/training_run'
ADAPTER_DIR     = f'{DRIVE_ROOT}/planner_lora_adapter'
JSONL_PATH      = f'{DRIVE_ROOT}/planner_training.jsonl'
CSV_BACKUP_PATH = f'{DRIVE_ROOT}/planner_pairs.csv'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ADAPTER_DIR, exist_ok=True)

# LoRA — Planner adapter (r=32 alpha=64 per doc)
LORA_R           = 32
LORA_ALPHA       = 64
LORA_DROPOUT     = 0.05
LORA_TARGET_MODS = ['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj']

# Training hyperparameters
MAX_SEQ_LEN  = 2048
EPOCHS       = 3
BATCH_SIZE   = 4
GRAD_ACCUM   = 4      # effective batch = 16
LR           = 1e-4
WARMUP_RATIO = 0.05

# Loss weight for NO examples (90/10 split — 2.5x makes 300 NO punch like 750)
NO_LOSS_WEIGHT = 2.5

# Tree-sitter: 10 files = ~640 tokens, comfortable within 2048
TS_MAX_FILES  = 10
TS_MAX_TOKENS = 640

SEED = 42
print('Config loaded.')

Config loaded.


In [9]:
# ============================================================
# CELL 4 — IMPORTS
# ============================================================
import json
import re
import csv as csv_module
import random
import warnings
from pathlib import Path

import pandas as pd
import torch
import snowflake.connector
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer
from datasets import Dataset
from sklearn.model_selection import train_test_split
from torch.nn import CrossEntropyLoss

warnings.filterwarnings('ignore')
random.seed(SEED)
csv_module.field_size_limit(10 * 1024 * 1024)

print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

assert Path(TREESITTER_INDEX_PATH).exists(), \
    f'treesitter_index.json not found at {TREESITTER_INDEX_PATH} — upload it before continuing'

PyTorch 2.10.0+cu128 | CUDA True
GPU: NVIDIA A100-SXM4-80GB


In [10]:
# ============================================================
# CELL 5 — LOAD DATA FROM CSV
# ============================================================
import itertools

yes_issues_df    = pd.read_csv(ISSUES_WITH_PRS_CSV)
no_issues_df = pd.read_csv(NO_PR_ISSUES_CSV)

yes_issues_df.columns    = [c.upper() for c in yes_issues_df.columns]
no_issues_df.columns = [c.upper() for c in no_issues_df.columns]

print(f'Joined rows (issue+PR pairs): {len(yes_issues_df)}')
print(f'Unique issues:               {yes_issues_df["ISSUE_NUMBER"].nunique()}')
print(f'No-PR issues:                {len(no_issues_df)}')

# Group consecutive rows by ISSUE_NUMBER — works because CSV is ordered
# by ISSUE_NUMBER ASC, CHANGED_FILES_COUNT ASC from the SQL export
issue_to_prs = {}
issue_meta   = {}

rows = yes_issues_df.to_dict('records')
for issue_num, group in itertools.groupby(rows, key=lambda r: r['ISSUE_NUMBER']):
    group_list = list(group)
    issue_to_prs[issue_num] = group_list
    issue_meta[issue_num] = {
        'ISSUE_NUMBER': issue_num,
        'TITLE':        group_list[0]['TITLE'],
        'ISSUE_RAW':    group_list[0]['ISSUE_RAW'],
    }

print(f'\nIssues with merged PRs: {len(issue_to_prs)}')
pr_counts = [len(v) for v in issue_to_prs.values()]
print(f'Avg PRs per issue: {sum(pr_counts)/len(pr_counts):.1f} | Max: {max(pr_counts)}')

Joined rows (issue+PR pairs): 2152
Unique issues:               1770
No-PR issues:                600

Issues with merged PRs: 1770
Avg PRs per issue: 1.2 | Max: 64


In [11]:
# ============================================================
# CELL 6 — LOAD TREE-SITTER INDEX AND HELPERS
#
# What the tree-sitter index does:
#   Provides the model with a condensed view of the codebase
#   so it can ground its file/function references in real symbols.
#   Heuristic retrieval: keyword-match issue text against file paths
#   and symbol names to get the most relevant 10 files.
#
# This same retrieval logic runs at production inference in the
# VSCode plugin. GraphRAG (Neo4j) is a Phase 2 enhancement that
# adds SIMILAR_PAST_FIXES context at inference without retraining.
#
# Noisy files excluded: auto-generated, test infrastructure,
# CLI tooling — these score high due to symbol count but are
# never the actual fix target.
# ============================================================

with open(TREESITTER_INDEX_PATH) as f:
    TS_INDEX = json.load(f)
print(f'Tree-sitter index: {len(TS_INDEX)} files')
total_syms = sum(len(v.get('classes',[])) + len(v.get('functions',[])) for v in TS_INDEX.values())
print(f'Total symbols: {total_syms}')

NOISY = ['generated.py', 'devel-common/', 'tests_common/', 'pytest_plugin', 'datamodels/', 'airflow-ctl/']

def is_noisy(fp): return any(p in fp.lower() for p in NOISY)

def extract_keywords(text: str) -> set:
    kws = set()
    kws.update(w.lower() for w in re.findall(r'\b[A-Z][a-zA-Z0-9]+\b', text))
    kws.update(re.findall(r'\b[a-z][a-z0-9]*(?:_[a-z0-9]+){1,}\b', text.lower()))
    kws.update(p.lower() for p in re.findall(r'airflow/[a-zA-Z0-9_/]+\.py', text))
    for kw in ['scheduler','executor','dag','task','operator','sensor','hook','provider',
               'trigger','xcom','variable','connection','webserver','api','model','database']:
        if kw in text.lower(): kws.add(kw)
    return kws

def get_ts_subset(text: str) -> dict:
    kws = extract_keywords(text)
    if not kws: return {}
    scored = []
    for fp, syms in TS_INDEX.items():
        if is_noisy(fp): continue
        score  = sum(2 for kw in kws if kw in fp.lower())
        score += sum(1 for sym in syms.get('classes',[]) + syms.get('functions',[])
                     for kw in kws if kw in sym.lower())
        if score > 0: scored.append((score, fp, syms))
    scored.sort(reverse=True, key=lambda x: x[0])
    return {fp: syms for _, fp, syms in scored[:TS_MAX_FILES]}

def format_ts(subset: dict) -> str:
    lines = []
    for fp, syms in subset.items():
        parts = []
        if syms.get('classes'):   parts.append(f"classes: {', '.join(syms['classes'][:5])}")
        if syms.get('functions'): parts.append(f"functions: {', '.join(syms['functions'][:8])}")
        if parts: lines.append(f"{fp} | {' | '.join(parts)}")
    return '\n'.join(lines)

def trunc(text: str, max_tokens: int) -> str:
    mc = max_tokens * 4
    return text[:mc] + '...' if len(text) > mc else text

# Quick sanity test
test_sub  = get_ts_subset('Scheduler crashes when DAG has more than 50 tasks')
test_text = format_ts(test_sub)
print(f'\nSanity test — {len(test_sub)} files, ~{len(test_text)//4} tokens')
print('Top 3:', list(test_sub.keys())[:3])

Tree-sitter index: 1957 files
Total symbols: 17121

Sanity test — 10 files, ~688 tokens
Top 3: ['task-sdk/src/airflow/sdk/execution_time/comms.py', 'airflow-core/src/airflow/jobs/scheduler_job_runner.py', 'airflow-core/src/airflow/models/dagrun.py']


In [12]:
# ============================================================
# CELL 7 — LOAD TOKENIZER
# ============================================================
print(f'Loading tokenizer: {BASE_MODEL}')
TOKENIZER = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
TOKENIZER.pad_token      = TOKENIZER.eos_token
TOKENIZER.padding_side   = 'right'

def count_tokens(text: str) -> int:
    return len(TOKENIZER.encode(text, add_special_tokens=False))

print('Tokenizer ready.')

Loading tokenizer: Qwen/Qwen2.5-Coder-7B-Instruct


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer ready.


In [13]:
# ============================================================
# CELL 8 — SPOT-CHECK NO EXAMPLES
# Review before building pairs — should be docs/questions/duplicates
# ============================================================
print('SPOT CHECK: 10 NO examples — confirm these are not bug fixes')
print('=' * 65)
for i, row in enumerate(no_issues_df.sample(10, random_state=SEED).itertuples()):
    raw    = json.loads(row.ISSUE_RAW)
    issue  = raw.get('issue', {})
    labels = [l.get('name','') for l in issue.get('labels',[])][:4]
    sr     = issue.get('state_reason', 'none')
    print(f'[{i+1:2}] #{row.ISSUE_NUMBER}: {str(row.TITLE)[:72]}')
    print(f'     labels={labels} | state_reason={sr}\n')

SPOT CHECK: 10 NO examples — confirm these are not bug fixes
[ 1] #57802: Expanding task sdk integration tests to test variable get operation
     labels=[] | state_reason=None

[ 2] #56282: Deprecate CreateAutoMLVideoTrainingJobOperator in google provider
     labels=['provider:google', 'area:providers', 'kind:documentation'] | state_reason=None

[ 3] #60762: Config: raise clear error for empty enum config values
     labels=[] | state_reason=None

[ 4] #57804: Update create_airflow_connection and delete_airflow_connection helper me
     labels=['provider:google', 'area:providers'] | state_reason=None

[ 5] #62617: Add self-review checklist for AI agents in AGENTS.md
     labels=[] | state_reason=None

[ 6] #58012: Enable PT006 rule to airflow-core tests (route/public)
     labels=['area:API'] | state_reason=None

[ 7] #59476: Check team boundaries in connections
     labels=['provider:google', 'provider:microsoft-azure', 'provider:amazon', 'area:providers'] | state_reason=None

[ 8] 

In [14]:
# ============================================================
# CELL 9 — BUILD YES TRAINING PAIRS
#
# ARCHITECTURE (important to understand):
#   INPUT  = issue text + tree-sitter context
#            NO PR content in input — matches production exactly
#            (at inference orchestrator sends live issue + TS context)
#   OUTPUT = structured plan derived from merged PR(s)
#            Root cause  ← first commit message of smallest PR
#            What to change ← concatenated bodies of all linked merged PRs
#                             (truncated, past→future reframe via prefix)
#            Target files ← accumulated unique files across all PRs
#                           DYNAMIC cap: keep as many files as fit in 2048
#                           not a hard cap — short-body issues get more files
#
# LABELS included in input: yes — help model distinguish bug vs feature
# Assignees/milestone/flags: dropped — no semantic planning value
#
# DYNAMIC FILE CAP explained:
#   Build output with full file list first
#   If combined input+output exceeds 2048 tokens, trim files one at a time
#   This preserves more ground truth for short-body issues
#   while fitting long-body issues within the limit
# ============================================================

def build_input_prompt(issue: dict, comments: list, ts_text: str) -> str:
    """Construct the model input — identical format used at inference."""
    title   = (issue.get('title') or '').strip()
    body    = trunc((issue.get('body') or '').strip(), 800)
    labels  = ', '.join(l.get('name','') for l in issue.get('labels',[]) if l.get('name'))
    cmts    = ' | '.join(
        (c.get('body') or '').strip()[:300]
        for c in comments if (c.get('body') or '').strip()
    )
    return (
        'You are a senior software engineer planning a code patch for an open GitHub issue.\n\n'
        f'ISSUE: {title}\n'
        f'LABELS: {labels or "none"}\n'
        f'BODY: {body}\n'
        f'COMMENTS: {cmts[:600] if cmts else "none"}\n'
        f'REPO_SYMBOLS:\n{ts_text}\n\n'
        'Decide if a code change is needed. If yes, produce a structured patch plan.'
    )


def build_plan_output(root_cause: str, what_to_change: str, files: list) -> str:
    """Construct the target plan output from PR-derived fields."""
    file_list_str = '\n  - '.join(files)
    return (
        'REQUIRES_CODE_CHANGE: YES\n'
        'CONFIDENCE: HIGH\n'
        f'REASON: {root_cause}\n\n'
        'PLAN:\n'
        f'- What to change: {what_to_change}\n'
        f'- Target files:\n  - {file_list_str}\n'
        '- Test strategy: Update or add unit tests for the modified functions.'
    )


def build_full_text(input_prompt: str, output: str) -> str:
    return (
        f'<|im_start|>user\n{input_prompt}<|im_end|>\n'
        f'<|im_start|>assistant\n{output}<|im_end|>'
    )


def build_yes_pair(issue_row: dict, prs_for_issue: list) -> dict | None:
    try:
        issue_raw = json.loads(issue_row['ISSUE_RAW'])
    except Exception:
        return None

    issue    = issue_raw.get('issue', {})
    comments = issue_raw.get('comments', [])

    # ---- INPUT ----
    title   = (issue.get('title') or '').strip()
    ts_text = format_ts(get_ts_subset(f"{title} {(issue.get('body') or '')[:400]}"))
    ts_text = trunc(ts_text, TS_MAX_TOKENS)
    input_prompt = build_input_prompt(issue, comments, ts_text)

    # ---- OUTPUT CONSTRUCTION FROM PRs ----
    # Root cause: first commit message of smallest PR (developer's own words)
    root_cause = ''
    for pr_row in prs_for_issue:
        try:
            pr_raw   = json.loads(pr_row['PR_RAW'])
            commits  = pr_raw.get('commits', [])
            if commits:
                root_cause = (commits[0].get('commit',{}).get('message') or '').strip().split('\n')[0]
                if root_cause: break
        except Exception:
            continue

    # What to change: concatenated PR bodies reframed as intent
    # Prefix "To fix this issue:" converts past-tense PR bodies to
    # future/instructional framing expected by the orchestrator
    pr_bodies = []
    for pr_row in prs_for_issue:
        try:
            pr_raw = json.loads(pr_row['PR_RAW'])
            body   = (pr_raw.get('pr', {}).get('body') or '').strip()
            if body:
                pr_bodies.append(body)
        except Exception:
            continue

    if not pr_bodies:
        return None

    # Concatenate bodies, truncate combined to 250 tokens
    combined_body  = ' | '.join(pr_bodies)
    what_to_change = 'To fix this issue: ' + trunc(combined_body.replace('\n', ' '), 250)

    # Accumulate unique files across ALL linked merged PRs
    all_files  = []
    seen_files = set()
    for pr_row in prs_for_issue:
        try:
            pr_raw = json.loads(pr_row['PR_RAW'])
            for f in pr_raw.get('files', []):
                fname = f.get('filename', '')
                if fname and fname not in seen_files:
                    all_files.append(fname)
                    seen_files.add(fname)
        except Exception:
            continue

    if not all_files:
        return None

    # ---- DYNAMIC FILE CAP ----
    # Build full output then trim files one at a time until within 2048
    # This preserves more ground truth for issues with short bodies
    files_to_use = all_files.copy()
    target_output = build_plan_output(root_cause, what_to_change, files_to_use)
    full_text     = build_full_text(input_prompt, target_output)
    tc            = count_tokens(full_text)

    while tc > MAX_SEQ_LEN and len(files_to_use) > 1:
        files_to_use  = files_to_use[:-1]   # drop last file
        target_output = build_plan_output(root_cause, what_to_change, files_to_use)
        full_text     = build_full_text(input_prompt, target_output)
        tc            = count_tokens(full_text)

    # If still over limit even with 1 file, truncate input body further
    if tc > MAX_SEQ_LEN:
        ts_text      = trunc(ts_text, 300)
        input_prompt = build_input_prompt(
            {**issue, 'body': trunc((issue.get('body') or ''), 400)},
            comments, ts_text
        )
        full_text = build_full_text(input_prompt, target_output)
        tc        = count_tokens(full_text)
        if tc > MAX_SEQ_LEN:
            return None   # discard — cannot fit even with maximum truncation

    return {
        'issue_number':  issue_row['ISSUE_NUMBER'],
        'pr_count':      len(prs_for_issue),
        'total_files':   len(files_to_use),
        'label':         'YES',
        'input':         input_prompt,
        'output':        target_output,
        'full_text':     full_text,
        'token_count':   tc,
    }


print('Building YES pairs...')
yes_pairs = []
skipped   = 0

for _, issue_row in yes_issues_df.iterrows():
    prs = issue_to_prs.get(issue_row['ISSUE_NUMBER'], [])
    if not prs:
        skipped += 1
        continue
    pair = build_yes_pair(issue_row.to_dict(), prs)
    if pair:
        yes_pairs.append(pair)
    else:
        skipped += 1
    if len(yes_pairs) % 200 == 0 and yes_pairs:
        print(f'  {len(yes_pairs)} YES pairs (skipped {skipped})...')
    if len(yes_pairs) >= YES_TARGET:
        break

random.shuffle(yes_pairs)
print(f'\nYES pairs: {len(yes_pairs)} | skipped: {skipped}')
multi_pr = sum(1 for p in yes_pairs if p['pr_count'] > 1)
print(f'Pairs drawing from multiple PRs: {multi_pr}')
avg_files = sum(p['total_files'] for p in yes_pairs) / max(len(yes_pairs), 1)
print(f'Avg files in output: {avg_files:.1f} (dynamic cap in effect)')

Building YES pairs...
  200 YES pairs (skipped 2)...
  400 YES pairs (skipped 2)...
  600 YES pairs (skipped 3)...
  800 YES pairs (skipped 6)...
  1000 YES pairs (skipped 10)...
  1200 YES pairs (skipped 13)...
  1400 YES pairs (skipped 15)...
  1600 YES pairs (skipped 17)...
  1800 YES pairs (skipped 18)...
  2000 YES pairs (skipped 18)...

YES pairs: 2132 | skipped: 20
Pairs drawing from multiple PRs: 558
Avg files in output: 5.7 (dynamic cap in effect)


In [15]:
# ============================================================
# CELL 10 — BUILD NO TRAINING PAIRS
# Same input format as YES — model sees identical prompt structure
# Output stops at REQUIRES_CODE_CHANGE: NO (no PLAN block)
# ============================================================

def build_no_pair(row: dict) -> dict | None:
    try:
        issue_raw = json.loads(row['ISSUE_RAW'])
    except Exception:
        return None

    issue    = issue_raw.get('issue', {})
    comments = issue_raw.get('comments', [])
    title    = (issue.get('title') or '').strip()

    ts_text      = format_ts(get_ts_subset(f"{title} {(issue.get('body') or '')[:400]}"))
    ts_text      = trunc(ts_text, TS_MAX_TOKENS)
    input_prompt = build_input_prompt(issue, comments, ts_text)

    label_str    = ' '.join(l.get('name','') for l in issue.get('labels',[])).lower()
    state_reason = issue.get('state_reason', '')

    if any(k in label_str for k in ['doc','question','duplicate']):
        reason = 'Issue requests documentation, clarification, or is a duplicate — no code defect present.'
    elif state_reason == 'not_planned':
        reason = 'Issue was closed as not planned — no code change was implemented.'
    else:
        reason = 'Issue was resolved without a code change — likely configuration, documentation, or support.'

    target_output = f'REQUIRES_CODE_CHANGE: NO\nCONFIDENCE: HIGH\nREASON: {reason}'
    full_text     = build_full_text(input_prompt, target_output)
    tc            = count_tokens(full_text)

    if tc > MAX_SEQ_LEN:
        return None

    return {
        'issue_number': row['ISSUE_NUMBER'],
        'pr_count':     0,
        'total_files':  0,
        'label':        'NO',
        'input':        input_prompt,
        'output':       target_output,
        'full_text':    full_text,
        'token_count':  tc,
    }


print('Building NO pairs...')
no_pairs = []
for _, row in no_issues_df.iterrows():
    if len(no_pairs) >= NO_TARGET: break
    p = build_no_pair(row.to_dict())
    if p: no_pairs.append(p)

print(f'NO pairs: {len(no_pairs)}')

Building NO pairs...
NO pairs: 300


In [16]:
# ============================================================
# CELL 11 — ASSEMBLE, VALIDATE AND SAVE
# ============================================================
import csv as csv_builtin

all_pairs = yes_pairs + no_pairs
random.shuffle(all_pairs)

n_yes = sum(1 for p in all_pairs if p['label'] == 'YES')
n_no  = sum(1 for p in all_pairs if p['label'] == 'NO')
print(f'Total: {len(all_pairs)} | YES: {n_yes} ({n_yes/len(all_pairs)*100:.1f}%) | NO: {n_no}')

# Token distribution
tc = sorted(p['token_count'] for p in all_pairs)
print(f'Tokens: min={tc[0]} median={tc[len(tc)//2]} p90={tc[int(len(tc)*0.9)]} max={tc[-1]}')

# Hard assert — no example over limit
over = [p['issue_number'] for p in all_pairs if p['token_count'] > MAX_SEQ_LEN]
assert len(over) == 0, f'Token overflow in {len(over)} examples: {over[:3]}'
print('Token limit check: PASS')

# Closure field leakage check
LEAK = ['closed_at', 'state_reason', 'closed_by', '"state": "closed"']
leaks = [p['issue_number'] for p in all_pairs if any(s in p['input'] for s in LEAK)]
assert len(leaks) == 0, f'Closure leaks in: {leaks[:5]}'
print('Closure leak check: PASS')

# Print sample YES pair for visual inspection
sy = next(p for p in all_pairs if p['label'] == 'YES')
print(f'\nSAMPLE YES — issue #{sy["issue_number"]} | {sy["pr_count"]} PRs | {sy["total_files"]} files | {sy["token_count"]}t')
print('INPUT (first 500 chars):')
print(sy['input'][:500])
print('\nOUTPUT:')
print(sy['output'])

# Print sample NO pair
sn = next(p for p in all_pairs if p['label'] == 'NO')
print(f'\nSAMPLE NO — issue #{sn["issue_number"]} | {sn["token_count"]}t')
print('OUTPUT:', sn['output'])

# Save JSONL to Drive
with open(JSONL_PATH, 'w') as f:
    for p in all_pairs:
        f.write(json.dumps({'text': p['full_text']}) + '\n')
print(f'\nJSONL saved: {JSONL_PATH}')

# Save CSV backup to Drive
with open(CSV_BACKUP_PATH, 'w', newline='') as f:
    w = csv_builtin.DictWriter(f, fieldnames=[
        'issue_number','label','pr_count','total_files','token_count','input_preview','output'
    ])
    w.writeheader()
    for p in all_pairs:
        w.writerow({
            'issue_number':  p['issue_number'],
            'label':         p['label'],
            'pr_count':      p['pr_count'],
            'total_files':   p['total_files'],
            'token_count':   p['token_count'],
            'input_preview': p['input'][:300].replace('\n', ' '),
            'output':        p['output'].replace('\n', ' | '),
        })
print(f'CSV backup saved: {CSV_BACKUP_PATH}')

Total: 2432 | YES: 2132 (87.7%) | NO: 300
Tokens: min=565 median=1349 p90=1975 max=2048
Token limit check: PASS
Closure leak check: PASS

SAMPLE YES — issue #61717 | 1 PRs | 2 files | 1573t
INPUT (first 500 chars):
You are a senior software engineer planning a code patch for an open GitHub issue.

ISSUE: [v3-1-test] Add XCom serilizer for pendulum.date.Date (#61176)
LABELS: type:misc/internal, area:DAG-processing
BODY: (cherry picked from commit c36678f9a5a664200d85b266ebd5e3d94d3baea4)

<!--
Thank you for contributing!

Please provide above a brief description of the changes made in this pull request.
Write a good git commit message following this guide: http://chris.beams.io/posts/git-commit/

Pl

OUTPUT:
REQUIRES_CODE_CHANGE: YES
CONFIDENCE: HIGH
REASON: Add XCom serilizer for pendulum.date.Date

PLAN:
...
- Target files:
  - task-sdk/src/airflow/sdk/serde/serializers/datetime.py
  - task-sdk/tests/task_sdk/serde/test_serializers.py
- Test strategy: Update or add unit tests for the 

In [17]:
# ============================================================
# CELL 12 — TRAIN / EVAL SPLIT (90/10, stratified by label)
# ============================================================
train_pairs, eval_pairs = train_test_split(
    all_pairs, test_size=0.10,
    stratify=[p['label'] for p in all_pairs],
    random_state=SEED
)
print(f'Train: {len(train_pairs)} | Eval: {len(eval_pairs)}')
print(f'Train YES/NO: {sum(1 for p in train_pairs if p["label"]=="YES")} / {sum(1 for p in train_pairs if p["label"]=="NO")}')

train_ds = Dataset.from_list([{'text': p['full_text']} for p in train_pairs])
eval_ds  = Dataset.from_list([{'text': p['full_text']} for p in eval_pairs])
print('Datasets ready.')

Train: 2188 | Eval: 244
Train YES/NO: 1918 / 270
Datasets ready.


In [18]:
# ============================================================
# CELL 13 — LOAD MODEL AND APPLY LORA
#
# Standard LoRA on bfloat16 — NOT QLoRA
# A100 80GB: 7B model uses ~14GB, 66GB headroom — no quantization needed
# QLoRA saves VRAM we don't need and adds gradient noise
#
# Only ~1-2% of parameters trained (LoRA adapter matrices)
# Base weights frozen — preserves Qwen's code generation capability
# ============================================================
print(f'Loading {BASE_MODEL}...')
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16,
    device_map='auto', trust_remote_code=True
)
model.config.use_cache = False

model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA,
    target_modules=LORA_TARGET_MODS,
    lora_dropout=LORA_DROPOUT,
    bias='none', task_type=TaskType.CAUSAL_LM
))
model.print_trainable_parameters()
# Expected: trainable ~1-2% — e.g. 'trainable params: 83M || all params: 7.7B || 1.09%'

if torch.cuda.is_available():
    print(f'VRAM allocated: {torch.cuda.memory_allocated()/1e9:.1f}GB')

Loading Qwen/Qwen2.5-Coder-7B-Instruct...


`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

trainable params: 60,555,264 || all params: 7,676,171,776 || trainable%: 0.7889
VRAM allocated: 15.5GB


In [19]:
# ============================================================
# CELL 14 — WEIGHTED LOSS TRAINER
#
# Applies 2.5x loss weight to NO examples
# Compensates for 90/10 YES/NO imbalance without oversampling
# 300 NO examples × 2.5 weight = equivalent signal of ~750 examples
# Better than 50/50 oversampling which would dilute YES signal
# ============================================================

class WeightedSFTTrainer(SFTTrainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        decoded = self.processing_class.batch_decode(inputs['input_ids'], skip_special_tokens=True)
        weights = [NO_LOSS_WEIGHT if 'REQUIRES_CODE_CHANGE: NO' in t else 1.0 for t in decoded]

        outputs    = model(**inputs)
        sl         = outputs.logits[..., :-1, :].contiguous()
        tl         = inputs['labels'][..., 1:].contiguous()
        per_token  = CrossEntropyLoss(reduction='none', ignore_index=-100)(
                         sl.view(-1, sl.size(-1)), tl.view(-1)
                     ).view(tl.size())
        w          = torch.tensor(weights, dtype=per_token.dtype, device=per_token.device).unsqueeze(1)
        mask       = (tl != -100).float()
        loss       = (per_token * w * mask).sum() / mask.sum().clamp(min=1)
        return (loss, outputs) if return_outputs else loss

print('WeightedSFTTrainer ready.')

WeightedSFTTrainer ready.


In [20]:
# ============================================================
# CELL 15 — TRAINING ARGUMENTS AND RUN
# Checkpoints every 100 steps saved to Drive
# Best model loaded at end based on eval loss
# Estimated: 10-14 hours on A100
# ============================================================


from trl import SFTConfig

args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    warmup_steps=20,
    lr_scheduler_type='cosine',
    weight_decay=0.01,
    bf16=True,
    gradient_checkpointing=True,
    eval_strategy='steps', eval_steps=100,
    save_strategy='steps', save_steps=100,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model='eval_loss',
    logging_steps=25,
    report_to='none',
    seed=SEED,
    remove_unused_columns=False,
    max_length=MAX_SEQ_LEN,
    packing=False,
    dataset_text_field='text',
)

trainer = WeightedSFTTrainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=eval_ds,
    processing_class=TOKENIZER,
)

spe = len(train_pairs) // (BATCH_SIZE * GRAD_ACCUM)
print(f'Train examples: {len(train_pairs)} | Steps/epoch: ~{spe} | Total steps: ~{spe*EPOCHS}')
print('Starting training...')
trainer.train()


Adding EOS to train dataset:   0%|          | 0/2188 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/2188 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2188 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/244 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/244 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/244 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151645}.


Train examples: 2188 | Steps/epoch: ~136 | Total steps: ~408
Starting training...


Step,Training Loss,Validation Loss
100,2.709227,0.642980
200,2.138649,0.574985
300,1.880545,0.554127
400,1.914300,0.549970


TrainOutput(global_step=411, training_loss=2.6847907388877403, metrics={'train_runtime': 3931.6414, 'train_samples_per_second': 1.67, 'train_steps_per_second': 0.105, 'total_flos': 5.024179926463488e+17, 'train_loss': 2.6847907388877403})

In [21]:
# ============================================================
# CELL 16 — SAVE ADAPTER AND QUICK INFERENCE TEST
# ============================================================
model.save_pretrained(ADAPTER_DIR)
TOKENIZER.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved: {ADAPTER_DIR}')
for fn in os.listdir(ADAPTER_DIR):
    print(f'  {fn}: {os.path.getsize(os.path.join(ADAPTER_DIR, fn))/1e6:.1f}MB')

# ---- Quick inference test ----
test   = eval_pairs[0]
prompt = f'<|im_start|>user\n{test["input"]}<|im_end|>\n<|im_start|>assistant\n'
ids    = TOKENIZER.encode(prompt, return_tensors='pt').to(model.device)

model.eval()
with torch.no_grad():
    out = model.generate(
        ids, max_new_tokens=400,
        do_sample=False,
        pad_token_id=TOKENIZER.eos_token_id
    )
generated = TOKENIZER.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

print(f'Test issue #{test["issue_number"]}')
print(f'Expected:\n{test["output"]}')
print(f'\nGenerated:\n{generated}')
print('\n--- FORMAT CHECKS ---')
print(f'Starts with REQUIRES_CODE_CHANGE: {"PASS" if generated.startswith("REQUIRES_CODE_CHANGE") else "FAIL"}')
print(f'Has PLAN or NO decision: {"PASS" if "PLAN:" in generated or "REQUIRES_CODE_CHANGE: NO" in generated else "FAIL"}')
print(f'No closure leakage: {"PASS" if not any(s in generated for s in ["closed_at","state_reason"]) else "FAIL"}')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Adapter saved: /content/drive/MyDrive/298B_WB2/planner_lora_adapter
  README.md: 0.0MB
  adapter_model.safetensors: 242.3MB
  adapter_config.json: 0.0MB
  chat_template.jinja: 0.0MB
  tokenizer_config.json: 0.0MB
  tokenizer.json: 11.4MB
Test issue #62781
Expected:
REQUIRES_CODE_CHANGE: YES
CONFIDENCE: HIGH
REASON: Variable table handle long words break when values are expanded

PLAN:
 <img width="1918" height="729" alt="Screenshot 2026-02-24 at 17 32 53" src="https://github.com/user-attachments/assets/378b3f59-aeb9-4008-a378-5f714da53438" />
- Target files:
  - airflow-core/src/airflow/ui/src/pages/Variables/Variables.tsx
- Test strategy: Update or add unit tests for the modified functions.

Generated:
REQUIRES_CODE_CHANGE: YES
CONFIDENCE: HIGH
REASON: Variable table handle long words break when values are expanded

PLAN:
 <img width="1824" height="1008" alt="image" src="https://github.com/user-attachments/assets/6e4f3b6a-ec0d-4510-b510-100000000001" />
- Target files:
  - airflow-cor